In [1]:
# install presidio via pip if not yet installed

#!pip install presidio-analyzer
#!pip install presidio-evaluator

In [2]:
import datetime
import pprint
from collections import Counter

import numpy as np
import pandas as pd

from presidio_evaluator import InputSample
from presidio_evaluator.data_generator import PresidioSentenceFaker

# Generate fake PII data using the Presidio Sentence Faker

The Presidio Sentence Faker enables you to generate a synthetic dataset from sentence templates.
Example templates:

> I live at {{address}}

> You can email me at {{email}}. Thanks, {{first_name}}

> What's your last name? It's {{last_name}}

> Every time I see you falling I get down on my knees and pray

### Simple example
This uses the default generator to create 10 samples based on three templates

In [3]:
sentence_templates = [
    "My name is {{name}}",
    "Please send it to {{address}}",
    "I just moved to {{city}} from {{country}}",
]


sentence_faker = PresidioSentenceFaker(
    "en_GB", lower_case_ratio=0.05, sentence_templates=sentence_templates
)
fake_sentence_results = sentence_faker.generate_new_fake_sentences(10)

# Print the spans of the first sample
print(fake_sentence_results[0].masked)
print(fake_sentence_results[0].spans)

Using default entity providers


Using default entity mapping between the entities in the templates and the ones in the output dataset
Using default provider aliases


Sampling:   0%|          | 0/10 [00:00<?, ?it/s]

Sampling: 100%|██████████| 10/10 [00:00<00:00, 6806.73it/s]

My name is {{PERSON}}
[Span(type: PERSON, value: Nestor Yermolayev, char_span: [11: 28])]


## Generate a full dataset

In this example we generate a large dataset with multiple entity types and save it in in JSON and CONLL03 formats.
This uses car finance support chat templates (`raw_data/car_finance_templates.txt`) instead of the default ones.

In [4]:
number_of_samples = 1500
lower_case_ratio = 0.05
locale = "en"
cur_time = datetime.date.today().strftime("%B_%d_%Y")

output_file = f"../data/generated_size_{number_of_samples}_date_{cur_time}.json"
output_conll = f"../data/generated_size_{number_of_samples}_date_{cur_time}.tsv"

The `PresidioSentenceFaker` is based on the Faker library. It loads [FakeNameGenerator](https://www.fakenamegenerator.com/) data by default
to extend the set of fake values and creates a `SentenceFaker` 
which returns a fake person record (with multiple values) instead of one value,
allowing dependencies between values belonging to the same fake person
(e.g. name = Michael Smith with the email michael.smith@gmail.com).

`FakeNameGenerator.com_3000.csv` is included in this package and can be sourced from https://www.fakenamegenerator.com/order.php

UK-specific entities are not covered by the default providers, so we add them here
with a custom `UkProvider`, mapped as follows:

| Template entity | Output entity | Format |
|---|---|---|
| `uk_phone_number` | `PHONE_NUMBER` | UK mobile / landline, national or +44 |
| `uk_driving_licence` | `UK_DRIVING_LICENCE` | 16-character DVLA licence number |
| `uk_address` | `STREET_ADDRESS` | One-line address with a valid-format postcode |
| `uk_postcode` | `STREET_ADDRESS` | Valid-format postcode, e.g. `SW1 2AB` |

Sentences come from car finance support chat templates (`car_finance_templates.txt`).

#### Optional: use real person records from the Snowflake DWH

By default, person records (names, addresses, phone numbers, ...) come from FakeNameGenerator.
Set `use_dwh_records = True` to instead sample real customers from `source_user.users_pii`
(full name, phone, email, full address, postcode, city, country). Values missing in the DWH fall back to Faker.

Requires `uv sync --extra snowflake` and `SNOWFLAKE_ACCOUNT` / `SNOWFLAKE_USER` / `SNOWFLAKE_ROLE` / `SNOWFLAKE_WAREHOUSE`
(env vars or `.env`; authentication defaults to `externalbrowser`).

⚠️ The generated dataset then contains real PII: keep it under `../data` (gitignored), and clear notebook outputs before committing.

In [5]:
use_dwh_records = True
dwh_records_limit = 3000

base_records = None  # None -> FakeNameGenerator records
if use_dwh_records:
    from presidio_evaluator.data_generator.faker_extensions.snowflake_records import (
        load_snowflake_person_df,
    )

    base_records = load_snowflake_person_df(limit=dwh_records_limit)
    # Let the UK templates below draw address/phone from the same DWH person
    base_records["uk_address"] = base_records["address"]
    base_records["uk_phone_number"] = base_records["phone_number"]
    base_records["uk_postcode"] = base_records["postcode"]
    print(f"Loaded {len(base_records)} person records from Snowflake")

/Users/oleksii.nikitchuk/Documents/carmoola/presidio-research/.venv/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Loaded 3000 person records from Snowflake


In [6]:
import random

from faker import Faker
from faker.providers import BaseProvider

from presidio_evaluator.data_generator import raw_data_dir

_gb_faker = Faker("en_GB")


class UkProvider(BaseProvider):
    POSTCODE_AREAS = [
        "B",
        "E",
        "G",
        "L",
        "M",
        "N",
        "S",
        "W",
        "AB",
        "BS",
        "BT",
        "CB",
        "CF",
        "EC",
        "EH",
        "LS",
        "NG",
        "NW",
        "OX",
        "SE",
        "SW",
        "WC",
    ]
    POSTCODE_INWARD_LETTERS = "ABDEFGHJLNPQRSTUWXYZ"

    def uk_phone_number(self):
        return self.numerify(
            random.choice(
                [
                    "07### ######",
                    "07#########",
                    "+44 7### ######",
                    "+447#########",
                    "020 #### ####",
                    "0161 ### ####",
                    "01### ######",
                    "+44 20 #### ####",
                ]
            )
        )

    def uk_driving_licence(self):
        # DVLA format: surname (5, padded with 9), birth decade, month (+50 for female),
        # day, birth year digit, initials (2, padded with 9), arbitrary digit, 2 check letters
        surname = "".join(c for c in _gb_faker.last_name().upper() if c.isalpha())[
            :5
        ].ljust(5, "9")
        birth_date = _gb_faker.date_of_birth(minimum_age=17, maximum_age=90)
        month = birth_date.month + (50 if random.random() < 0.5 else 0)
        initials = _gb_faker.first_name()[0].upper() + random.choice(
            [_gb_faker.first_name()[0].upper(), "9"]
        )
        return (
            f"{surname}{str(birth_date.year)[2]}{month:02d}{birth_date.day:02d}{str(birth_date.year)[3]}"
            f"{initials}9{self.lexify('??', letters='ABCDEFGHJKLMNPRSTUVWXYZ')}"
        )

    def uk_postcode(self):
        outward = random.choice(self.POSTCODE_AREAS) + str(random.randint(1, 20))
        inward = str(random.randint(0, 9)) + self.lexify(
            "??", letters=self.POSTCODE_INWARD_LETTERS
        )
        return f"{outward} {inward}"

    def uk_address(self):
        street = f"{_gb_faker.building_number()} {_gb_faker.street_name()}"
        if random.random() < 0.3:
            street = f"Flat {random.randint(1, 30)}, {street}"
        return f"{street}, {_gb_faker.city()}, {self.uk_postcode()}"


sentence_templates = [
    line.strip()
    for line in (raw_data_dir / "car_finance_templates.txt").read_text().splitlines()
    if line.strip()
]

entity_type_mapping = {
    **PresidioSentenceFaker.ENTITY_TYPE_MAPPING,
    "uk_phone_number": "PHONE_NUMBER",
    "uk_driving_licence": "UK_DRIVING_LICENCE",
    "uk_address": "STREET_ADDRESS",
    "uk_postcode": "STREET_ADDRESS",
    "date": "DATE_TIME",
}

sentence_faker = PresidioSentenceFaker(
    "en_GB",
    lower_case_ratio=0.05,
    sentence_templates=sentence_templates,
    entity_type_mapping=entity_type_mapping,
    base_records=base_records,
    # With DWH records, never fall back to Faker for entities the DWH provides
    fill_from_other_records=use_dwh_records,
)
sentence_faker.add_provider(UkProvider)

Using default entity providers


Using default provider aliases


In [7]:
pd.DataFrame(sentence_faker._sentence_faker.records).head()

,person,phone_number,email,address,postcode,city,country,first_name,last_name,name,...,name_male,first_name_female,first_name_male,first_name_nonbinary,last_name_female,last_name_male,zipcode,uk_address,uk_phone_number,uk_postcode
0,Jordan Guy,+447508404894,jordguy9@gmail.com,"22 Steam Mill Lane, RIPLEY, DE5 3JR",DE5 3JR,RIPLEY,England,Jordan,Guy,Jordan Guy,...,Jordan Guy,Jordan,Jordan,Jordan,Guy,Guy,DE5 3JR,"22 Steam Mill Lane, RIPLEY, DE5 3JR",+447508404894,DE5 3JR
1,Liz Mcginnes,NaN,lizmcginnes1@hotmail.com,"11 Marsh Drive, Beverley, HU17 8WD",HU17 8WD,Beverley,England,Liz,Mcginnes,Liz Mcginnes,...,Liz Mcginnes,Liz,Liz,Liz,Mcginnes,Mcginnes,HU17 8WD,"11 Marsh Drive, Beverley, HU17 8WD",NaN,HU17 8WD
2,Dashamir Rexhaj,+447473136003,dashamirrexhaj129@gmail.com,"13 Ridley close, Barking, IG11 9PJ",IG11 9PJ,Barking,England,Dashamir,Rexhaj,Dashamir Rexhaj,...,Dashamir Rexhaj,Dashamir,Dashamir,Dashamir,Rexhaj,Rexhaj,IG11 9PJ,"13 Ridley close, Barking, IG11 9PJ",+447473136003,IG11 9PJ
3,Troy Mclean,NaN,mrice6969@yahoo.co.uk,"2 Farnsworth house Knights hill, London, SE27 0EL",SE27 0EL,London,England,Troy,Mclean,Troy Mclean,...,Troy Mclean,Troy,Troy,Troy,Mclean,Mclean,SE27 0EL,"2 Farnsworth house Knights hill, London, SE27 0EL",NaN,SE27 0EL
4,Chidinma Aku,+447547237215,Chidinmaaku46@gmail.com,"FLAT 6 6 13 HONEYSUCKLE WALK, BRISTOL, BS10 6FR",BS10 6FR,BRISTOL,England,Chidinma,Aku,Chidinma Aku,...,Chidinma Aku,Chidinma,Chidinma,Chidinma,Aku,Aku,BS10 6FR,"FLAT 6 6 13 HONEYSUCKLE WALK, BRISTOL, BS10 6FR",+447547237215,BS10 6FR


`PresidioSentenceFaker` adds additional providers by default, which are not included in the Faker package.
These can be found in `presidio_evaluator.data_generator.faker_extensions.providers`

It is possible to create providers for additional entity types by extending Faker's `BaseProvider` class, 
and calling `add_provider` on the `PresidioSentenceFaker` instance.
For example:

In [8]:
import random

from faker.providers import BaseProvider


class MarsIdProvider(BaseProvider):
    def mars_id(self):
        # Generate a random row number between 1 and 50
        row = random.randint(1, 50)
        # Generate a random letter for the seat location from A-K
        location = random.choice("ABCDEFGHIJK")
        # Return the seat in the format "row-letter" (e.g., "25A")
        return f"{row}{location}"


sentence_faker.add_provider(MarsIdProvider)
# Now a new `mars_id` entity can be generated if a template has `mars_id` in it.

In [9]:
from presidio_evaluator.data_generator.faker_extensions.providers import *

IpAddressProvider  # Both Ipv4 and IPv6 IP addresses
NationalityProvider  # Read countries + nationalities from file
OrganizationProvider  # Read organization names from file
UsDriverLicenseProvider  # Read US driver license numbers from file
AgeProvider  # Age values (unavailable on Faker
AddressProviderNew  # Extend the default address formats
PhoneNumberProviderNew  # Extend the default phone number formats
ReligionProvider  # Read religions from file

presidio_evaluator.data_generator.faker_extensions.providers.ReligionProvider

`PresidioSentenceFaker.PROVIDER_ALIASES` can be extended to add additional provider aliases for when templates have
a different entity name than what the providers emit.

In [10]:
# Create entity aliases (e.g. if your provider supports "name" but templates contain "person").
provider_aliases = PresidioSentenceFaker.PROVIDER_ALIASES
provider_aliases

# To customize, call `PresidioSentenceFaker(locale="en_US",...,provider_aliases=provider_aliases)`

[('name', 'person'),
 ('credit_card_number', 'credit_card'),
 ('date_of_birth', 'birthday')]

Generate data

In [11]:
fake_records = sentence_faker.generate_new_fake_sentences(num_samples=number_of_samples)
pprint.pprint(fake_records[0])

Sampling:   0%|          | 0/1500 [00:00<?, ?it/s]

Sampling:  61%|██████    | 909/1500 [00:00<00:00, 9089.19it/s]

Sampling: 100%|██████████| 1500/1500 [00:00<00:00, 9439.12it/s]

Full text: My employer is Synchronoss Technologies Inc., I've been there for 8 years
Spans: [Span(type: AGE, value: 8, char_span: [66: 67]), Span(type: ORGANIZATION, value: Synchronoss Technologies Inc., char_span: [15: 44])]



#### Verify randomness of dataset

In [12]:
count_per_template_id = Counter([sample.template_id for sample in fake_records])

print(f"Total: {sum(count_per_template_id.values())}")
print(f"Avg # of records per template: {np.mean(list(count_per_template_id.values()))}")
print(
    f"Median # of records per template: {np.median(list(count_per_template_id.values()))}"
)
print(f"Std: {np.std(list(count_per_template_id.values()))}")

Total: 1500
Avg # of records per template: 20.0
Median # of records per template: 20.0
Std: 4.958494395143214


#### Which entities did we generate?

In [13]:
count_per_entity = Counter()
for record in fake_records:
    count_per_entity.update(Counter([span.entity_type for span in record.spans]))

count_per_entity

Counter({'PERSON': 656,
         'DATE_TIME': 273,
         'STREET_ADDRESS': 258,
         'PHONE_NUMBER': 208,
         'EMAIL_ADDRESS': 187,
         'ORGANIZATION': 163,
         'UK_DRIVING_LICENCE': 126,
         'CREDIT_CARD': 67,
         'TITLE': 45,
         'GPE': 38,
         'IBAN_CODE': 34,
         'AGE': 21})

In [14]:
for record in fake_records[:10]:
    print(record)

Full text: My employer is Synchronoss Technologies Inc., I've been there for 8 years
Spans: [Span(type: AGE, value: 8, char_span: [66: 67]), Span(type: ORGANIZATION, value: Synchronoss Technologies Inc., char_span: [15: 44])]

Full text: I haven't had the confirmation email, is it going to kafaya.kalib@hotmail.com?
Spans: [Span(type: EMAIL_ADDRESS, value: kafaya.kalib@hotmail.com, char_span: [53: 77])]

Full text: Hi there, it's Caglar, I'm just chasing up my application for the Ford Focus
Spans: [Span(type: PERSON, value: Caglar, char_span: [15: 21])]

Full text: Please don't call me on +447718884725 anymore, email me at shamsul_alom786@hotmail.co.uk instead
Spans: [Span(type: EMAIL_ADDRESS, value: shamsul_alom786@hotmail.co.uk, char_span: [59: 88]), Span(type: PHONE_NUMBER, value: +447718884725, char_span: [24: 37])]

Full text: Can you send the documents in the post to 62  Thistlebarrow Road, Salisbury, SP1 3RS?
Spans: [Span(type: STREET_ADDRESS, value: 62  Thistlebarrow Road, Salis

#### Save as json

In [15]:
InputSample.to_json(dataset=fake_records, output_file=output_file)

In [16]:
output_file

'../data/generated_size_1500_date_October_23_2026.json'

#### Create a CONLL like data frame

In [17]:
conll = InputSample.create_conll_dataset(dataset=fake_records)
conll.head(10)

  0%|          | 0/1500 [00:00<?, ?it/s]

loading model en_core_web_sm


  0%|          | 1/1500 [00:00<06:04,  4.11it/s]

  2%|▏         | 23/1500 [00:00<00:17, 83.95it/s]

  3%|▎         | 41/1500 [00:00<00:12, 115.75it/s]

...

 95%|█████████▌| 1426/1500 [00:05<00:00, 257.61it/s]

 97%|█████████▋| 1453/1500 [00:06<00:00, 259.18it/s]

 99%|█████████▊| 1479/1500 [00:06<00:00, 258.30it/s]

100%|██████████| 1500/1500 [00:06<00:00, 241.61it/s]

,text,pos,tag,template_id,label,sentence
0,My,PRON,PRP$,37,O,0
1,employer,NOUN,NN,37,O,0
2,is,AUX,VBZ,37,O,0
3,Synchronoss,PROPN,NNP,37,B-ORGANIZATION,0
4,Technologies,PROPN,NNPS,37,I-ORGANIZATION,0
5,Inc.,PROPN,NNP,37,I-ORGANIZATION,0
6,",",PUNCT,",",37,O,0
7,I,PRON,PRP,37,O,0
8,'ve,AUX,VBP,37,O,0
9,been,AUX,VBN,37,O,0


In [18]:
conll.to_csv(output_conll, sep="\t")
print(f"CoNLL2003 dataset structure output location: {output_conll}")

CoNLL2003 dataset structure output location: ../data/generated_size_1500_date_October_23_2026.tsv


In [19]:
# from ollama import chat

# response = chat(
#     model='gemma4:latest',
#     messages=[{'role': 'user', 'content': 'act as presidio PII analyzer and analyse the text:```I\'d like it to be sent to 669 Sonnenweg 23, Narsaq, Greenlander```'}]
# )

# print(response.message.content)

### Next steps

- Evaluate Presidio using fake data: [Sample](4_Evaluate_Presidio_Analyzer.ipynb)
- Split to train/test/validation while ensuring sentences originiating from the same template are all on the same subset: [Sample](3_Split_by_pattern_#.ipynb)
- Conduct a small exploratory data analysis on the generated data: [Sample](2_PII_EDA.ipynb)

#### Copyright notice:


Data generated for evaluation was created using Fake Name Generator.

Fake Name Generator identities by the [Fake Name Generator](https://www.fakenamegenerator.com/) 
are licensed under a [Creative Commons Attribution-Share Alike 3.0 United States License](http://creativecommons.org/licenses/by-sa/3.0/us/). Fake Name Generator and the Fake Name Generator logo are trademarks of Corban Works, LLC.